In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rasaang/homework-3/01_Naive Bayes Classifier for Diabetes Prediction.ipynb.txt
/kaggle/input/datasets/rasaang/homework-3/Homework 3 LAB-assignment.txt
/kaggle/input/datasets/rasaang/homework-3/Naive Bayes Classifier for Diabetes Prediction.ipynb
/kaggle/input/datasets/rasaang/homework-3/Naive Bayes Classifier for Diabetes Prediction.ipynb.txt
/kaggle/input/datasets/rasaang/homework-3/diabetes.csv
/kaggle/input/datasets/rasaang/acript-hw-3-naive-bayes-classifier/naive bayes Homework 3 script.py


In [2]:
# Rasaan Gouldbourne - AI 240
# Homework 3: Naive Bayes Classifier for Diabetes Prediction
# Instructor: Dr. Uma Gajendragadkar

import os
import sys
import warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)
sns.set_theme(style="whitegrid")

INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def step_1_read_data(input_dir: str) -> pd.DataFrame:
    """Step 1: Scan /kaggle/input, list files, and load the Diabetes CSV dataset."""
    print("=" * 60, flush=True)
    print("STEP 1: SCANNING /kaggle/input FOR DIABETES CSV", flush=True)
    print("=" * 60, flush=True)

    csv_path = None
    all_found_files = []

    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            for file in files:
                full_path = os.path.join(root, file)
                all_found_files.append(full_path)
                # Ignore .ipynb or .txt notebook templates; look for CSV or tabular data with 'Outcome'
                if not file.lower().endswith('.ipynb') and not file.lower().endswith('.ipynb.txt'):
                    try:
                        sample_df = pd.read_csv(full_path, nrows=2)
                        cols = [c.strip() for c in sample_df.columns]
                        if 'Outcome' in cols:
                            csv_path = full_path
                            break
                    except Exception:
                        continue
            if csv_path:
                break

    print("Files detected in /kaggle/input:", flush=True)
    for f in all_found_files:
        print(f"  -> {f}", flush=True)

    if csv_path and os.path.exists(csv_path):
        print(f"\nLoaded Diabetes Dataset: {csv_path}", flush=True)
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(
            "Could not find a dataset file containing the 'Outcome' column in /kaggle/input. "
            "Check the file list printed above to verify your CSV file is uploaded."
        )

    df.columns = [col.strip() for col in df.columns]
    return df


def step_2_explore_data(df: pd.DataFrame, output_dir: str) -> None:
    """Step 2: Display dataset info, print 10 rows, and save exploration charts."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 2: DATASET INFO, FIRST 10 ROWS & EXPLORATION CHARTS", flush=True)
    print("=" * 60, flush=True)

    print("\n--- 2a) Dataset Info ---", flush=True)
    df.info()

    print("\n--- 2b) First 10 Rows of Dataset ---", flush=True)
    print(df.head(10), flush=True)

    # Exploration Chart 1: Target Class Distribution
    plt.figure(figsize=(6, 4))
    sns.countplot(x='Outcome', hue='Outcome', data=df, palette='Set2', legend=False)
    plt.title('Distribution of Diabetes Outcome (0 = Negative, 1 = Positive)')
    plt.xlabel('Outcome')
    plt.ylabel('Patient Count')
    plt.tight_layout()
    dist_path = os.path.join(output_dir, 'step2_outcome_distribution.png')
    plt.savefig(dist_path, dpi=300)
    plt.close()
    print(f"\nSaved Exploration Chart 1: {dist_path}", flush=True)

    # Exploration Chart 2: Feature Correlation Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Diabetes Features Correlation Heatmap')
    plt.tight_layout()
    corr_path = os.path.join(output_dir, 'step2_correlation_heatmap.png')
    plt.savefig(corr_path, dpi=300)
    plt.close()
    print(f"Saved Exploration Chart 2: {corr_path}", flush=True)


def step_3_and_4_remove_pregnancies(df: pd.DataFrame) -> pd.DataFrame:
    """Step 3 & Step 4: Remove Pregnancies column, display info, and check shape."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 3 & 4: REMOVING 'PREGNANCIES' COLUMN & CHECKING SHAPE", flush=True)
    print("=" * 60, flush=True)

    if 'Pregnancies' in df.columns:
        df = df.drop(columns=['Pregnancies'])
        print("Successfully removed 'Pregnancies' column.", flush=True)

    print("\n--- Updated Dataset Info ---", flush=True)
    df.info()
    print(f"\nUpdated Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns", flush=True)
    return df


def step_5_clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Step 5: Clean data by handling zero-encoded missing medical values."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 5: CLEANING DATA", flush=True)
    print("=" * 60, flush=True)

    df_clean = df.copy()
    print("Missing (NaN) values before cleaning:", flush=True)
    print(df_clean.isnull().sum(), flush=True)

    clinical_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    cols_to_fix = [c for c in clinical_zero_cols if c in df_clean.columns]

    for col in cols_to_fix:
        zero_count = (df_clean[col] == 0).sum()
        if zero_count > 0:
            df_clean[col] = df_clean[col].replace(0, np.nan)
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            print(f"Imputed {zero_count} zero-entries in '{col}' using median.", flush=True)

    df_clean = df_clean.drop_duplicates().dropna()
    print(f"Clean dataset ready: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns", flush=True)
    return df_clean


def step_6_to_11_model_pipeline(df_clean: pd.DataFrame, output_dir: str) -> None:
    """Steps 6 to 11: Split data (80:20), train GaussianNB, predict, and evaluate."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 6: SPLITTING DATA INTO TRAINING & TESTING SETS (80:20)", flush=True)
    print("=" * 60, flush=True)

    X = df_clean.drop(columns=['Outcome'])
    y = df_clean['Outcome'].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}", flush=True)

    print("\n" + "=" * 60, flush=True)
    print("STEP 7 & 8: INITIALIZING & FITTING NAIVE BAYES MODEL", flush=True)
    print("=" * 60, flush=True)

    model = GaussianNB()
    model.fit(X_train_scaled, y_train)
    print("Gaussian Naive Bayes model initialized and fitted successfully.", flush=True)

    print("\n" + "=" * 60, flush=True)
    print("STEP 9, 10 & 11: PREDICTIONS, ACCURACY & CONFUSION MATRIX", flush=True)
    print("=" * 60, flush=True)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred)

    print(f"Model Testing Accuracy : {acc:.4f} ({acc * 100:.2f}%)", flush=True)
    print(f"Model ROC-AUC Score    : {roc_auc:.4f}", flush=True)
    print("\nClassification Report:", flush=True)
    print(report, flush=True)

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:", flush=True)
    print(cm, flush=True)

    report_path = os.path.join(output_dir, 'step10_11_evaluation_report.txt')
    with open(report_path, 'w') as f:
        f.write("Naive Bayes Classifier - Diabetes Prediction Evaluation\n")
        f.write("=" * 55 + "\n")
        f.write(f"Model Testing Accuracy : {acc:.4f} ({acc * 100:.2f}%)\n")
        f.write(f"Model ROC-AUC Score    : {roc_auc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(np.array2string(cm))
    print(f"\nSaved Evaluation Report: {report_path}", flush=True)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Naive Bayes Confusion Matrix (Diabetes Prediction)')
    plt.xlabel('Predicted Label (0 = No, 1 = Yes)')
    plt.ylabel('True Label (0 = No, 1 = Yes)')
    plt.tight_layout()
    cm_path = os.path.join(output_dir, 'step11_confusion_matrix.png')
    plt.savefig(cm_path, dpi=300)
    plt.close()
    print(f"Saved Confusion Matrix Plot: {cm_path}", flush=True)
    print("=" * 60, flush=True)


if __name__ == "__main__":
    df = step_1_read_data(INPUT_DIR)
    step_2_explore_data(df, OUTPUT_DIR)
    df = step_3_and_4_remove_pregnancies(df)
    df_clean = step_5_clean_data(df)
    step_6_to_11_model_pipeline(df_clean, OUTPUT_DIR)


STEP 1: SCANNING /kaggle/input FOR DIABETES CSV
Files detected in /kaggle/input:
  -> /kaggle/input/datasets/rasaang/homework-3/01_Naive Bayes Classifier for Diabetes Prediction.ipynb.txt
  -> /kaggle/input/datasets/rasaang/homework-3/Homework 3 LAB-assignment.txt
  -> /kaggle/input/datasets/rasaang/homework-3/Naive Bayes Classifier for Diabetes Prediction.ipynb
  -> /kaggle/input/datasets/rasaang/homework-3/Naive Bayes Classifier for Diabetes Prediction.ipynb.txt
  -> /kaggle/input/datasets/rasaang/homework-3/diabetes.csv

Loaded Diabetes Dataset: /kaggle/input/datasets/rasaang/homework-3/diabetes.csv

STEP 2: DATASET INFO, FIRST 10 ROWS & EXPLORATION CHARTS

--- 2a) Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 no